# Hypothesis Testing: Null, Alternative & p-values

Companion notebook for the [Hypothesis Testing wiki page](https://ml-viz-ruby.vercel.app/wiki/hypothesis-testing).

We build the logic of a significance test from scratch: simulate the null distribution of a test
statistic, see where the observed statistic falls, compute a p-value two ways (analytic + simulation),
reproduce the A/B-test worked example, and demonstrate why 'peeking' inflates the false-positive rate.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#ccc',
    'xtick.color':      '#888',
    'ytick.color':      '#888',
    'text.color':       '#eee',
    'grid.color':       '#333',
    'lines.linewidth':  2,
})

rng = np.random.default_rng(42)

## 1 — The null distribution and the p-value, visualized

Under $H_0$ a standardized test statistic $T$ follows the standard normal. The two-sided p-value is
the total probability mass in both tails beyond $|t_{\text{obs}}|$ — the shaded area below.

In [ ]:
t_obs = 1.96
x = np.linspace(-4, 4, 500)
pdf = stats.norm.pdf(x)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(x, pdf, color='#6366f1', label='null distribution N(0,1)')
ax.fill_between(x, pdf, where=np.abs(x) >= t_obs, color='#fb7185', alpha=0.6,
                label=f'p-value tails (|T| ≥ {t_obs})')
for t in (-t_obs, t_obs):
    ax.axvline(t, color='#fb7185', ls='--', lw=1)
ax.set_xlabel('test statistic T'); ax.set_ylabel('density')
ax.set_title('A two-sided p-value is the tail area beyond |t_obs| under H0')
ax.legend(facecolor='#1a1d27', edgecolor='#444')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

p = 2 * (1 - stats.norm.cdf(t_obs))
print(f"two-sided p-value at t_obs = {t_obs}:  {p:.4f}")

## 2 — The A/B-test worked example

Variant A: 200/2000 (10.0%). Variant B: 240/2000 (12.0%).
$H_0: p_A = p_B$ vs $H_1: p_A \ne p_B$ at $\alpha = 0.05$.

In [ ]:
def two_proportion_ztest(xA, nA, xB, nB):
    pA, pB = xA / nA, xB / nB
    p_pool = (xA + xB) / (nA + nB)
    se = np.sqrt(p_pool * (1 - p_pool) * (1 / nA + 1 / nB))
    z = (pB - pA) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    return z, p_value

z, p_value = two_proportion_ztest(200, 2000, 240, 2000)
print(f"z statistic = {z:.3f}")
print(f"p-value     = {p_value:.4f}")
print("reject H0 (B better)" if p_value < 0.05 else "fail to reject H0")

## 3 — A p-value is uniform under the null

A subtle but defining fact: **if $H_0$ is true, the p-value is uniformly distributed on $[0,1]$.**
That is *why* `P(p ≤ α | H0) = α` — the significance level is exactly the false-positive rate.
We verify by simulating thousands of A/A tests (two arms drawn from the *same* rate).

In [ ]:
n, true_rate, trials = 2000, 0.10, 20000
pvals = np.empty(trials)
for i in range(trials):
    a = rng.binomial(n, true_rate)
    b = rng.binomial(n, true_rate)        # same rate -> H0 is TRUE
    _, pvals[i] = two_proportion_ztest(a, n, b, n)

false_pos = np.mean(pvals < 0.05)
print(f"empirical P(p < 0.05 | H0) = {false_pos:.3f}   (target α = 0.05)")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(pvals, bins=20, color='#6366f1', edgecolor='#0f1117')
ax.axhline(trials / 20, color='#2dd4bf', ls='--', label='uniform expectation')
ax.set_xlabel('p-value'); ax.set_ylabel('count')
ax.set_title('Under H0 the p-value is uniform on [0, 1]')
ax.legend(facecolor='#1a1d27', edgecolor='#444')
plt.tight_layout(); plt.show()

## 4 — The peeking problem

Checking significance repeatedly and stopping at the first hit is multiple testing in disguise. We
run A/A experiments where the analyst peeks daily and stops early on any $p < 0.05$ — the
false-positive rate blows past the nominal 5%.

In [ ]:
def peeking_experiment(rng, daily=200, days=14, rate=0.10):
    """Run an A/A test, peeking each 'day'. Return True if we ever cross p<0.05."""
    a_tot = b_tot = na = nb = 0
    for _ in range(days):
        a_tot += rng.binomial(daily, rate); na += daily
        b_tot += rng.binomial(daily, rate); nb += daily
        _, p = two_proportion_ztest(a_tot, na, b_tot, nb)
        if p < 0.05:
            return True          # stopped early on a false positive
    return False

runs = 3000
peek_fp = np.mean([peeking_experiment(rng) for _ in range(runs)])
print(f"false-positive rate WITH daily peeking: {peek_fp:.3f}")
print("(vs the nominal 0.05 for a single look at a fixed sample size)")

## ✏️ Your turn

**Exercise.** Implement `decide(p_value, alpha)` returning the string `"reject H0"` when the result
is significant and `"fail to reject H0"` otherwise, and `power_simulation(rng, effect, n, alpha, trials)`
that estimates the **power** of the two-proportion test: the fraction of experiments that correctly
reject $H_0$ when variant B's true rate is `0.10 + effect` (so $H_0$ is genuinely false).

Power is `P(reject H0 | H1 true)` — estimate it by simulating `trials` experiments and counting how
often `p < alpha`.

In [ ]:
def decide(p_value, alpha=0.05):
    # TODO(you): return 'reject H0' or 'fail to reject H0'
    return ...

def power_simulation(rng, effect, n=2000, alpha=0.05, trials=4000):
    base = 0.10
    rejects = 0
    for _ in range(trials):
        a = rng.binomial(n, base)
        b = rng.binomial(n, base + effect)   # H1 is TRUE
        # TODO(you): run the test, increment `rejects` when p < alpha
        ...
    return rejects / trials

In [ ]:
# This assert cell passes silently when your implementation is correct.
assert decide(0.03) == "reject H0"
assert decide(0.20) == "fail to reject H0"

pw = power_simulation(np.random.default_rng(0), effect=0.02)
assert 0.4 < pw < 0.8, f"power for a 2-point lift at N=2000 should be moderate, got {pw:.2f}"
# Bigger effects are easier to detect -> higher power
assert power_simulation(np.random.default_rng(1), effect=0.05) > pw
print(f"✓ all checks passed   (power for a 2-point lift ≈ {pw:.2f})")

<details>
<summary>Solution</summary>

```python
def decide(p_value, alpha=0.05):
    return "reject H0" if p_value < alpha else "fail to reject H0"

def power_simulation(rng, effect, n=2000, alpha=0.05, trials=4000):
    base = 0.10
    rejects = 0
    for _ in range(trials):
        a = rng.binomial(n, base)
        b = rng.binomial(n, base + effect)
        _, p = two_proportion_ztest(a, n, b, n)
        if p < alpha:
            rejects += 1
    return rejects / trials
```

Power rises with the effect size, the sample size, and the significance level α. The whole point of
a sample-size calculation is to pick `n` so that power reaches a target (often 0.80) for the smallest
effect you care about (the MDE).

</details>